# Deployment-mode quality, recurrent drift, and Othello continuation

This notebook keeps four concepts separate:

- **Free-generation quality:** the task-specific success measure under `recompute` and `append_recurrent`.
- **Generation-position drift:** where legality first deteriorates along a generated suffix.
- **Teacher-forced schedule gap:** disagreement between inference schedules on the same gold prefix, before sampled-token errors compound.
- **Othello state knowledge:** probability assigned to the entire legal move set, which is distinct from matching one randomly chosen legal game.

Paired seed lines are more informative than overlapping bars. Evaluation throughput should only be compared when example count, device, batch size, and output-length distribution match.

In [ ]:
from collections import defaultdict
from pathlib import Path
from statistics import median
import sys

import matplotlib
if "ipykernel" in sys.modules:
    matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
REPO_ROOT = HERE if (HERE / "experiments").exists() else HERE.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from figures.plotting_utils import (
    ARCHITECTURE_COLORS,
    INFERENCE_MODE_STYLES,
    filter_records,
    grouped,
    load_diagnostic_records,
    load_drift_records,
    load_othello_examples,
    metric_label,
    primary_metric,
    set_plot_style,
    unique_values,
)

set_plot_style()
RESULT_ROOT = REPO_ROOT / "results"
FIGURE_DIR = REPO_ROOT / "figures"


In [ ]:
drift = load_drift_records(RESULT_ROOT)
diagnostics = load_diagnostic_records(RESULT_ROOT)
othello_examples = load_othello_examples(RESULT_ROOT)
print(f"drift summaries: {len(drift)}")
print(f"diagnostic summaries: {len(diagnostics)}")
print(f"Othello continuation rows: {len(othello_examples)}")
print("drift tasks:", unique_values(drift, "task"))


## Final quality and measured evaluation throughput

For shortest path, the primary measure is complete optimal-path generation; valid-edge rate is useful but insufficient. For Othello trace evaluation, token legality is the dense measure and full-sequence legality is the strict measure.

In [ ]:
TASK = "shortest_path"
DEVICE = None
SHORTEST_PATH_DISTRIBUTION = "main"
ARCHITECTURES = list(ARCHITECTURE_COLORS)
QUALITY_METRIC = primary_metric(TASK)

selected = filter_records(drift, task=TASK, device=DEVICE)
if TASK == "shortest_path":
    selected = filter_records(selected, shortest_path_distribution=SHORTEST_PATH_DISTRIBUTION)
selected = [row for row in selected if row.get("architecture") in ARCHITECTURES]
print(f"Selected {len(selected)} summaries; quality metric = {QUALITY_METRIC!r}")
summary_keys = defaultdict(list)
for row in selected:
    summary_keys[(row.get("architecture"), row.get("seed"), row.get("inference_mode"))].append(row["summary_path"])
duplicates = {key: paths for key, paths in summary_keys.items() if len(paths) > 1}
if duplicates:
    print("WARNING: duplicate architecture/seed/mode summaries found; narrow RESULT_ROOT before pairing:")
    for key, paths in duplicates.items():
        print(" ", key, *paths, sep="\n    ")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
mode_x = {"recompute": 0, "append_recurrent": 1}

for architecture_index, architecture in enumerate(ARCHITECTURES):
    color = ARCHITECTURE_COLORS.get(architecture)
    architecture_rows = [row for row in selected if row.get("architecture") == architecture]
    for (_seed,), seed_rows in grouped(architecture_rows, "seed").items():
        points = sorted(
            [(mode_x[row["inference_mode"]], row[QUALITY_METRIC]) for row in seed_rows
             if row.get("inference_mode") in mode_x and row.get(QUALITY_METRIC) is not None]
        )
        if points:
            jitter = (architecture_index - (len(ARCHITECTURES) - 1) / 2) * 0.025
            axes[0].plot([x + jitter for x, _ in points], [y for _, y in points], "o-", color=color, alpha=0.55)

    for mode, x in mode_x.items():
        values = [row[QUALITY_METRIC] for row in architecture_rows
                  if row.get("inference_mode") == mode and row.get(QUALITY_METRIC) is not None]
        if values:
            axes[0].scatter(x + (architecture_index - (len(ARCHITECTURES) - 1) / 2) * 0.025, median(values),
                            color=color, edgecolor="black", linewidth=0.5, s=70,
                            label=architecture if mode == "recompute" else None, zorder=5)

    throughput = [row.get("eval_output_tok_per_s") for row in architecture_rows
                  if row.get("inference_mode") == "append_recurrent" and row.get("eval_output_tok_per_s") is not None]
    if throughput:
        axes[1].scatter([architecture_index] * len(throughput), throughput, color=color, alpha=0.55)
        axes[1].hlines(median(throughput), architecture_index - 0.25, architecture_index + 0.25,
                       color="black", linewidth=2)

axes[0].set_xticks([0, 1], ["K-pass\nrecompute", "append\nrecurrent"])
axes[0].set_ylabel(metric_label(QUALITY_METRIC))
axes[0].set_ylim(-0.02, 1.02)
axes[0].set_title("Paired deployment-mode quality")
axes[0].legend(fontsize=8)
axes[1].set_xticks(range(len(ARCHITECTURES)), [name.replace("_", "\n") for name in ARCHITECTURES])
axes[1].set_ylabel("Generated output tokens / second")
axes[1].set_title(f"Append-recurrent throughput ({DEVICE or 'set DEVICE before comparing'})")
fig.suptitle(TASK, y=1.02)
fig.tight_layout()
# fig.savefig(FIGURE_DIR / f"{TASK}_deployment_quality.png", dpi=220, bbox_inches="tight")

## Where free generation fails

Path-step accuracy measures whether each generated transition matches the unique shortest path. This makes late-position collapse visible without introducing a separate legality score. Individual seeds remain visible; the heavy curve is the unsmoothed median at each step.

In [ ]:
POSITION_ARCHITECTURE = "memory_tape"
position_rows = [row for row in selected if row.get("architecture") == POSITION_ARCHITECTURE]

fig, ax = plt.subplots(figsize=(8.7, 4.6))
for mode in ("recompute", "append_recurrent"):
    mode_rows = [row for row in position_rows if row.get("inference_mode") == mode]
    values_by_position = defaultdict(list)
    for row in mode_rows:
        points = sorted((int(key.removeprefix("path_step_").removesuffix("_accuracy")), value)
                        for key, value in row.items()
                        if key.startswith("path_step_") and key.endswith("_accuracy"))
        if points:
            ax.plot(*zip(*points), color="tab:blue" if mode == "recompute" else "tab:orange",
                    linestyle=INFERENCE_MODE_STYLES[mode], alpha=0.18)
        for position, value in points:
            values_by_position[position].append(value)
    positions = sorted(values_by_position)
    if positions:
        ax.plot(positions, [median(values_by_position[position]) for position in positions],
                color="tab:blue" if mode == "recompute" else "tab:orange",
                linestyle=INFERENCE_MODE_STYLES[mode], linewidth=2.5, label=mode)
ax.set(xlabel="Path transition step", ylabel="Accuracy", ylim=(-0.02, 1.02),
       title=f"{TASK}: accuracy along generation ({POSITION_ARCHITECTURE})")
ax.legend()
fig.tight_layout()
# fig.savefig(FIGURE_DIR / f"{TASK}_{POSITION_ARCHITECTURE}_position_drift.png", dpi=220, bbox_inches="tight")

## Teacher-forced schedule gap

Here both schedules receive the same gold tokens. Positive NLL delta means the append schedule assigns less probability to the correct next token than exact recomputation. Memory distance is descriptive: a large distance matters only when it accompanies logit or quality deterioration.

In [ ]:
schedule_records = [row for row in diagnostics
                    if row.get("task") == TASK and row.get("architecture") == POSITION_ARCHITECTURE
                    and (DEVICE is None or row.get("device") == DEVICE)]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.1), sharex=True)
fields = [
    ("nll_delta", "Append − recompute NLL"),
    ("logit_kl", "Logit KL"),
    ("memory_rms_delta", "Memory RMS distance"),
]
for ax, (field, label) in zip(axes, fields):
    by_position = defaultdict(list)
    for row in schedule_records:
        positions = row["payload"]["teacher_forced_schedule_gap"]["positions"]
        points = [(item["generated_position"], item[field]) for item in positions if item.get("count", 0) > 0]
        if points:
            ax.plot(*zip(*points), color=ARCHITECTURE_COLORS.get(POSITION_ARCHITECTURE), alpha=0.2)
        for position, value in points:
            by_position[position].append(value)
    positions = sorted(by_position)
    if positions:
        ax.plot(positions, [median(by_position[position]) for position in positions],
                color=ARCHITECTURE_COLORS.get(POSITION_ARCHITECTURE), linewidth=2.5)
    ax.set(xlabel="Gold suffix position", ylabel=label)
    if field == "nll_delta":
        ax.axhline(0, color="black", linewidth=1)
fig.suptitle(f"{TASK}: teacher-forced inference-schedule divergence", y=1.03)
fig.tight_layout()
# fig.savefig(FIGURE_DIR / f"{TASK}_{POSITION_ARCHITECTURE}_schedule_gap.png", dpi=220, bbox_inches="tight")

## Othello random-prefix evaluation

The left panel measures whether free generation stays legal. The middle panel measures teacher-forced probability mass assigned to **all** legal moves, avoiding a penalty for preferring a different legal continuation. The right panel shows full legal termination, a deliberately strict end-to-end criterion. Stratification by prompt length tests whether a more informative prefix produces a more useful persistent state.

In [ ]:
OTHELLO_ARCHITECTURE = "memory_tape"
OTHELLO_PROTOCOL = "random-prefix"  # or prefix-grid-0.25, prefix-grid-0.5, prefix-grid-0.75, full-game
othello_selected = [row for row in othello_examples
                    if row.get("architecture") == OTHELLO_ARCHITECTURE
                    and row.get("protocol") == OTHELLO_PROTOCOL
                    and (DEVICE is None or row.get("device") == DEVICE)]
print(f"Selected {len(othello_selected)} Othello continuation rows")

In [ ]:
bucket_order = ["0", "1-15", "16-30", "31-45", "46+"]
panels = [
    ("free_generation.legal_move_fraction", "Free-generation legal moves"),
    ("teacher_forced.legal_probability_mass", "Teacher-forced legal probability mass"),
    ("free_generation.sequence_legality", "Fully legal terminal continuation"),
]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharex=True)
for ax, (field, title) in zip(axes, panels):
    for mode, color in (("recompute", "tab:blue"), ("append_recurrent", "tab:orange")):
        mode_rows = [row for row in othello_selected if row.get("inference_mode") == mode]
        means = []
        positions = []
        for index, bucket in enumerate(bucket_order):
            bucket_rows = [row for row in mode_rows if row.get("prompt_bucket") == bucket and row.get(field) is not None]
            if bucket_rows:
                positions.append(index)
                if field.startswith("teacher_forced."):
                    weights = [row.get("teacher_forced.move_count", 0.0) for row in bucket_rows]
                    total_weight = sum(weights)
                    means.append(sum(row[field] * weight for row, weight in zip(bucket_rows, weights)) / total_weight)
                else:
                    means.append(sum(row[field] for row in bucket_rows) / len(bucket_rows))
        if positions:
            ax.plot(positions, means, marker="o", color=color,
                    linestyle=INFERENCE_MODE_STYLES[mode], label=mode)
    ax.set_xticks(range(len(bucket_order)), bucket_order, rotation=25)
    ax.set_xlabel("Prompt moves")
    ax.set_ylabel(title)
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(title)
axes[0].legend()
fig.suptitle(f"Othello {OTHELLO_PROTOCOL}: {OTHELLO_ARCHITECTURE}", y=1.03)
fig.tight_layout()
# fig.savefig(FIGURE_DIR / f"othello_{OTHELLO_PROTOCOL}_{OTHELLO_ARCHITECTURE}.png", dpi=220, bbox_inches="tight")